[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/05_attention.ipynb)

# 🔴 Hard: Softmax Attention

Implement the core attention mechanism used in Transformers.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Signature
```python
def scaled_dot_product_attention(
    Q: torch.Tensor,  # (batch, seq_q, d_k)
    K: torch.Tensor,  # (batch, seq_k, d_k)
    V: torch.Tensor,  # (batch, seq_k, d_v)
) -> torch.Tensor:   # (batch, seq_q, d_v)
    ...
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- You **may** use `torch.softmax` and `torch.bmm`
- Must support autograd
- Must handle cross-attention (seq_q ≠ seq_k)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

/Users/EndUser/miniconda3/envs/cs229/lib/python3.12/site-packages/torch/nn/modules/transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


#bmm is matmul for batch

In [3]:
Q = torch.ones([2, 3, 4])
K = torch.ones([2, 4, 2])
# V = torch.ones([2, 4, 5])


In [4]:
A= torch.bmm(Q, K.T)

/var/folders/l6/9gdvb1qj4qn1zn_lvwc9c5wc0000gp/T/ipykernel_11083/1291312536.py:1: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:3641.)
  A= torch.bmm(Q, K.T)


In [5]:
V = torch.ones([2, 2, 5])

In [6]:
A.shape

torch.Size([2, 3, 2])

In [7]:
torch.bmm(A, V)

tensor([[[8., 8., 8., 8., 8.],
         [8., 8., 8., 8., 8.],
         [8., 8., 8., 8., 8.]],

        [[8., 8., 8., 8., 8.],
         [8., 8., 8., 8., 8.],
         [8., 8., 8., 8., 8.]]])

In [22]:
# ✏️ YOUR IMPLEMENTATION HERE

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = torch.bmm(Q, torch.transpose(K,1,2))/math.sqrt(d_k)
    attn = torch.softmax(scores, dim = -1)
    return torch.bmm(attn, V)

In [ ]:
Q.shape[-1]

AttributeError: 'int' object has no attribute 'dtype'

In [39]:
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

In [40]:
Q.shape


torch.Size([2, 4, 8])

In [37]:
K.shape

torch.Size([2, 4, 8])

In [34]:
K.T.shape

torch.Size([8, 4, 2])

In [36]:
torch.transpose(K, 1, 2).shape

torch.Size([2, 8, 4])

In [23]:
# 🧪 Debug
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)          # should be (2, 4, 8)
print("Has NaN?    ", torch.isnan(out).any().item())  # should be False
print("Has Inf?    ", torch.isinf(out).any().item())  # should be False

# Cross-attention: seq_q != seq_k
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attn shape:", out2.shape)     # should be (1, 3, 32)

Output shape: torch.Size([2, 4, 8])
Has NaN?     False
Has Inf?     False
Cross-attn shape: torch.Size([1, 3, 32])


In [24]:
# ✅ SUBMIT
from torch_judge import check
check("attention")


🧪 Testing: Softmax Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (3.0ms)
  ✅ [2/4] Numerical correctness (1.2ms)
  ✅ [3/4] Gradient check (3.7ms)
  ✅ [4/4] Cross-attention (seq_q != seq_k) (0.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (8.1ms total)
  Progress saved. Run status() to see your dashboard.

